# VietOCR fine-tune: bilingual enterprise print

Train and evaluate VietOCR on approved Vietnamese and English enterprise printed crops.

## 1. Install dependencies

Enable a Kaggle GPU (T4) and internet. The notebook logs the training stages; VietOCR prints loss every 100 steps and validation metrics every 250.

In [ ]:
import importlib.util, subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "vietocr==0.3.13"])
for module, package in (("imgaug","imgaug==0.4.0"),("prefetch_generator","prefetch_generator==1.0.1"),
    ("einops","einops==0.2.0"),("albumentations","albumentations==1.4.2"),("lmdb","lmdb"),
    ("gdown","gdown==4.4.0"),("rapidfuzz","rapidfuzz")):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",package])
import numpy as np
# Compatibility for imgaug 0.4 on Kaggle NumPy 2 / Python 3.13.
if not hasattr(np,"sctypes"):
    np.sctypes={"int":[np.int8,np.int16,np.int32,np.int64],"uint":[np.uint8,np.uint16,np.uint32,np.uint64],
        "float":[np.float16,np.float32,np.float64],"complex":[np.complex64,np.complex128],
        "others":[bool,object,bytes,str]}
print("VietOCR dependencies ready", flush=True)


## 2. Load and validate the shared split

Use corrected Vietnamese and English enterprise printed rows. Preserve source-PDF grouping; the held-out test never selects a checkpoint.


In [ ]:
import hashlib, json, random, shutil, unicodedata, zipfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from rapidfuzz.distance import Levenshtein
from vietocr.model.trainer import Trainer
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

INPUT_ROOT=Path('/kaggle/input'); PACKAGE=next(INPUT_ROOT.rglob('dataset-info.json')).parent
WORK=Path('/kaggle/working/enterprise-printed-ocr'); WORK.mkdir(parents=True,exist_ok=True)
DATA=WORK/'data'; DATA.mkdir(parents=True,exist_ok=True)
image_dir=PACKAGE/'images'
image_zip=next(PACKAGE.glob('images*.zip'),None)
if not (DATA/'images').exists():
    if image_dir.is_dir():
        (DATA/'images').symlink_to(image_dir,target_is_directory=True)
    elif image_zip:
        with zipfile.ZipFile(image_zip) as zf: zf.extractall(DATA)
    else:
        raise FileNotFoundError('Combined dataset has neither images/ nor images.zip')
for name in ('dataset-info.json','annotation_train.txt','annotation_validation.txt','annotation_test.txt','manifest_train.jsonl','manifest_validation.jsonl','manifest_test.jsonl'):
    shutil.copy2(PACKAGE/name,DATA/name)
info=json.loads((DATA/'dataset-info.json').read_text(encoding='utf-8'))
SEED=2026; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU before VietOCR fine-tuning')

def read_split(name):
    annotations=(DATA/f'annotation_{name}.txt').read_text(encoding='utf-8').splitlines()
    manifest=[json.loads(line) for line in (DATA/f'manifest_{name}.jsonl').read_text(encoding='utf-8').splitlines()]
    if len(annotations)!=len(manifest): raise ValueError(f'{name}: annotation/manifest count mismatch')
    rows=[]
    for line,record in zip(annotations,manifest):
        image,sep,text=line.partition('\t')
        if not sep or image!=record['image'] or text!=record['text']: raise ValueError(f'Bad {name} row: {line[:100]}')
        relative=Path(image)
        if relative.is_absolute() or '..' in relative.parts: raise ValueError(f'Unsafe image path: {image}')
        path=DATA/relative
        if not path.is_file() or not text.strip(): raise ValueError(f'Missing image or empty label: {image}')
        record=dict(record); record.update(path=path,text=text); rows.append(record)
    return rows
train_all=read_split('train'); val_all=read_split('validation'); test_all=read_split('test')
correction_counts={split:sum(bool(row.get('label_correction_applied')) for row in rows)
    for split,rows in [('train',train_all),('validation',val_all),('test',test_all)]}
declared_corrections=info.get('confirmed_transcript_corrections_applied',{})
if correction_counts!=declared_corrections:
    print(json.dumps({'warning':'dataset-info correction counts are stale; manifest counts will be used',
        'dataset_info':declared_corrections,'manifest':correction_counts},ensure_ascii=False),flush=True)
    info['confirmed_transcript_corrections_applied']=correction_counts
print('Verified materialized dataset corrections:',correction_counts,flush=True)
def group_set(rows,key): return {row[key] for row in rows if row.get(key) is not None}
for key in ('source_pdf_sha256',):
    assert not group_set(train_all,key)&group_set(val_all,key)
    assert not group_set(train_all,key)&group_set(test_all,key)
    assert not group_set(val_all,key)&group_set(test_all,key)
assert len(test_all)==info['counts']['test']['rows']
printed_domains={'enterprise_printed','enterprise_english'}
for split,rows in [('train',train_all),('validation',val_all),('test',test_all)]:
    unexpected={row['domain'] for row in rows}-printed_domains
    if unexpected: raise ValueError(f'Unexpected domains in {split}: {sorted(unexpected)}')
print(json.dumps({'experiment_domains':sorted(printed_domains),
    'printed_rows_before_vocab_filter':{'train':len(train_all),'validation':len(val_all),'test':len(test_all)}},
    ensure_ascii=False,indent=2))


## 3. Metrics

In [ ]:
import csv

PUNCTUATION_TRANSLATION = str.maketrans({
    '‘': "'", '’': "'", '‚': "'", '‛': "'",
    '“': '"', '”': '"', '„': '"', '‟': '"', '»': '"',
    '–': '-', '—': '-', '−': '-', '•': '*', '™': 'TM', '›': '>', '⁄': '/',
})
def normalize_whitespace(text):
    return ' '.join(unicodedata.normalize('NFC',str(text)).strip().split())
def normalize(text):
    return normalize_whitespace(str(text).translate(PUNCTUATION_TRANSLATION))
def metric_pair(refs,hyps):
    char_errors=sum(Levenshtein.distance(a,b) for a,b in zip(refs,hyps))
    word_errors=sum(Levenshtein.distance(a.split(),b.split()) for a,b in zip(refs,hyps))
    return {'CER':char_errors/max(sum(map(len,refs)),1),
        'WER':word_errors/max(sum(len(a.split()) for a in refs),1),
        'exact_match':sum(a==b for a,b in zip(refs,hyps))/max(len(refs),1)}
def score(name,rows,predictions):
    refs=[normalize(row['text']) for row in rows]; hyps=[normalize(text) for text in predictions]
    raw_refs=[normalize_whitespace(row.get('reference_raw',row['text'])) for row in rows]
    raw_hyps=[normalize_whitespace(text) for text in predictions]
    normalized_metrics=metric_pair(refs,hyps); raw_metrics=metric_pair(raw_refs,raw_hyps)
    return {'system':name,'n_lines':len(refs),**normalized_metrics,
        'raw_CER':raw_metrics['CER'],'raw_WER':raw_metrics['WER'],'raw_exact_match':raw_metrics['exact_match']}
def save_prediction_audit(path, split, rows, predictions_by_model):
    for name, predictions in predictions_by_model.items():
        if len(predictions) != len(rows):
            raise ValueError(f"{name} predictions have {len(predictions)} rows; expected {len(rows)}")
    fields = ['split', 'domain', 'language', 'source', 'source_pdf_sha256', 'page',
              'source_image', 'image', 'reference_raw',
              'reference'] + [f'{name}_prediction' for name in predictions_by_model]
    path = Path(path)
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        for index, row in enumerate(rows):
            record = {key: row.get(key, '') for key in fields}
            record.update({
                'split': split,
                'image': row.get('image', row.get('path', '')),
                'reference': row.get('text', ''),
                'reference_raw': row.get('reference_raw', row.get('text', '')),
            })
            for name, predictions in predictions_by_model.items():
                record[f'{name}_prediction'] = predictions[index]
            writer.writerow(record)
    print(f'Saved {split} prediction audit: {len(rows)} rows to {path}', flush=True)

def metrics_by_domain(name,rows,predictions):
    result=[score(name+' / all',rows,predictions)]
    for domain in ('enterprise_printed','enterprise_english'):
        indices=[i for i,row in enumerate(rows) if row['domain']==domain]
        result.append(score(name+' / '+domain,[rows[i] for i in indices],[predictions[i] for i in indices]))
    return result
def predict_vietocr(config,rows,stage):
    predictor=Predictor(config); output=[]
    for i,row in enumerate(rows,1):
        with Image.open(row['path']) as image: output.append(predictor.predict(image.convert('RGB')))
        if i%200==0 or i==len(rows): print(f'{stage}: {i}/{len(rows)}',flush=True)
    return output


## 4. Fine-tune `vgg_seq2seq`

Use LR `1e-4`, batch size 32, freeze CNN for 300 steps, then unfreeze in 250-step blocks up to 3,000 total steps. Early-stop after two checks without improvement in the unweighted mean of validation WER for Vietnamese print and English print; restore the best printed-domain checkpoint. Test data never selects a checkpoint.

In [ ]:
base_config=Cfg.load_config_from_name('vgg_seq2seq'); base_config['device']='cuda:0'
model_vocab=set(base_config['vocab'])
def supported_rows(rows):
    kept=[]; excluded=[]
    for original in rows:
        row={**original,'reference_raw':original['text'],'text':normalize(original['text'])}
        (kept if set(row['text'])<=model_vocab else excluded).append(row)
    return kept,excluded
train_rows,train_excluded=supported_rows(train_all)
val_rows,val_excluded=supported_rows(val_all)
test_rows,test_excluded=supported_rows(test_all)
print({'train_used':len(train_rows),'train_oov_excluded':len(train_excluded),'validation_used':len(val_rows),
       'validation_oov_excluded':len(val_excluded),'test_used_for_both_engines':len(test_rows),'test_oov_excluded_for_both':len(test_excluded)})

DATA_WORK=WORK/'vietocr-data'
if DATA_WORK.exists(): shutil.rmtree(DATA_WORK)
DATA_WORK.mkdir(parents=True)
def write_annotations(rows,split):
    lines=[]
    for i,row in enumerate(rows):
        # Absolute paths point into the read-only Kaggle input; Trainer builds its LMDB directly from them.
        lines.append(f'{row["path"]}\t{row["text"]}')
        if (i+1)%1000==0 or i+1==len(rows):
            print(f'Prepared VietOCR {split} annotations: {i+1}/{len(rows)}',flush=True)
    # VietOCR 0.3.13's LMDB writer stores count-1; this sentinel preserves all unique rows.
    lines.append(lines[-1])
    (DATA_WORK/f'annotation_{split}.txt').write_text('\n'.join(lines)+'\n',encoding='utf-8')
write_annotations(train_rows,'train'); write_annotations(val_rows,'validation')

config=Cfg.load_config_from_name('vgg_seq2seq'); config['device']='cuda:0'
config['dataset'].update(data_root=str(DATA_WORK),train_annotation='annotation_train.txt',valid_annotation='annotation_validation.txt')
config['optimizer'].update(max_lr=1e-4,pct_start=0.1)
weights_path=WORK/'vietocr-vgg-seq2seq-enterprise-print.pth'; phase1=WORK/'phase1-frozen-300.pth'; best_path=WORK/'best-validation-wer.pth'
config['trainer'].update(batch_size=32,iters=3000,print_every=100,valid_every=10**9,
    checkpoint=str(WORK/'vietocr-checkpoint.pth'),export=str(phase1),log=str(WORK/'vietocr-train.log'))
config_path=WORK/'vietocr-config.json'; config_path.write_text(json.dumps(config,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
trainer=Trainer(config)
from torch.utils.data import DataLoader
valid_dataset=trainer.valid_gen.dataset; valid_collate=trainer.valid_gen.collate_fn
trainer.valid_gen=DataLoader(valid_dataset,batch_size=1,shuffle=False,collate_fn=valid_collate,drop_last=False,**config['dataloader'])
validation_seen=sum(len(batch['filenames']) for batch in trainer.valid_gen)
if validation_seen!=len(val_rows): raise RuntimeError(f'Validation loader saw {validation_seen}/{len(val_rows)} rows')
validation_by_path={str(row['path']):row for row in val_rows}
def validation_domain_wer():
    trainer.model.eval()
    try:
        predictions,references,filenames,_=trainer.predict()
    finally:
        trainer.model.train()
    domain_refs={'enterprise_printed':[],'enterprise_english':[]}
    domain_hyps={'enterprise_printed':[],'enterprise_english':[]}
    for filename,reference,prediction in zip(filenames,references,predictions):
        row=validation_by_path.get(str(filename))
        if row and row['domain'] in domain_refs:
            domain_refs[row['domain']].append(normalize(reference))
            domain_hyps[row['domain']].append(normalize(prediction))
    domain_wer={domain:metric_pair(domain_refs[domain],domain_hyps[domain])['WER']
        for domain in domain_refs if domain_refs[domain]}
    if set(domain_wer)!={'enterprise_printed','enterprise_english'}:
        raise RuntimeError(f'Validation WER missing enterprise domain: {domain_wer}')
    return domain_wer,sum(domain_wer.values())/len(domain_wer)

trainer.model.cnn.freeze(); trainer.num_iters=300; trainer.export_weights=str(phase1)
print('Starting frozen-CNN phase: 300 steps',flush=True)
trainer.train()
best_domain_wer,best_validation_wer=validation_domain_wer()
best_step=trainer.iter; stale_validations=0; validation_history=[]
validation_history.append({'step':best_step,'domain_wer':best_domain_wer,'macro_validation_wer':best_validation_wer})
trainer.save_weights(str(best_path))
print({'step':best_step,'domain_wer':best_domain_wer,'macro_validation_wer':best_validation_wer},flush=True)
trainer.model.cnn.unfreeze()
for target_step in (500,750,1000,1250,1500,1750,2000,2250,2500,2750,3000):
    trainer.num_iters=target_step-trainer.iter
    trainer.train()
    domain_wer,macro_wer=validation_domain_wer()
    validation_history.append({'step':trainer.iter,'domain_wer':domain_wer,'macro_validation_wer':macro_wer})
    print({'step':trainer.iter,'domain_wer':domain_wer,'macro_validation_wer':macro_wer},flush=True)
    if macro_wer < best_validation_wer-0.001:
        best_domain_wer,best_validation_wer=domain_wer,macro_wer
        best_step=trainer.iter; stale_validations=0
        trainer.save_weights(str(best_path))
    else:
        stale_validations+=1
        if stale_validations>=2:
            print(f'Early stopping at step {trainer.iter}; best step {best_step}',flush=True)
            break
trainer.load_weights(str(best_path)); trainer.save_weights(str(weights_path))
if not weights_path.is_file(): raise FileNotFoundError('VietOCR weights missing')


## 5. Evaluate pretrained and fine-tuned VietOCR

Test metrics do not select the checkpoint.

In [ ]:
baseline_config=Cfg.load_config_from_name('vgg_seq2seq'); baseline_config['device']='cuda:0'
ft_config=Cfg.load_config_from_name('vgg_seq2seq'); ft_config['device']='cuda:0'; ft_config['weights']=str(weights_path)
base_test_preds=predict_vietocr(baseline_config,test_rows,'VietOCR pretrained test')
ft_test_preds=predict_vietocr(ft_config,test_rows,'VietOCR fine-tuned test')
val_preds=predict_vietocr(ft_config,val_rows,'VietOCR validation')
save_prediction_audit(WORK/'vietocr-validation-predictions.csv','validation',val_rows,{'fine_tuned':val_preds})
save_prediction_audit(WORK/'vietocr-test-predictions.csv','test',test_rows,{'pretrained':base_test_preds,'fine_tuned':ft_test_preds})
vietocr_results=metrics_by_domain('VietOCR pretrained',test_rows,base_test_preds)
vietocr_results+=metrics_by_domain('VietOCR fine-tuned',test_rows,ft_test_preds)
validation_results=metrics_by_domain('VietOCR fine-tuned / validation',val_rows,val_preds)
print(json.dumps({'validation':validation_results,'test':vietocr_results},ensure_ascii=False,indent=2))


## 6. Save and upload VietOCR artifacts

Upload weights, config, and aggregate metrics only. No images, transcripts, or per-line predictions are sent to Hugging Face.

In [ ]:
import hashlib

metrics={'dataset':info,'label_corrections_applied':info.get('confirmed_transcript_corrections_applied',{}),'text_normalization':'NFC; collapse Unicode whitespace; canonicalize curly quotes and dash variants to ASCII; map bullets to *, trademark to TM, and fraction slash to /','architecture':'vgg_seq2seq','experiment_scope':'enterprise printed only; Vietnamese and English','freeze_cnn_steps':300,'unfreeze_cnn_steps':1200,
    'max_lr':1e-4,'batch_size':32,'selected_step':best_step,'best_validation_macro_domain_wer':best_validation_wer,'validation_history':validation_history,
    'printed_rows_before_vocab_filter':{'train':len(train_all),'validation':len(val_all),'test':len(test_all)},
    'train_rows_used':len(train_rows),'train_rows_oov_excluded':len(train_excluded),
    'validation_rows_used':len(val_rows),'validation_rows_oov_excluded':len(val_excluded),
    'test_rows_used':len(test_rows),'test_rows_oov_excluded':len(test_excluded),
    'validation_metrics':validation_results,'test_metrics':vietocr_results,
    'prediction_audit_files':['vietocr-validation-predictions.csv','vietocr-test-predictions.csv']}
metrics_path=WORK/'vietocr-metrics.json'
metrics_path.write_text(json.dumps(metrics,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
metadata_path=WORK/'vietocr-run-metadata.json'
metadata_path.write_text(json.dumps({'dataset_id':'Enterprise Printed OCR Lines',
    'weights_sha256':hashlib.sha256(weights_path.read_bytes()).hexdigest(),
    'config_sha256':hashlib.sha256(config_path.read_bytes()).hexdigest(),'metrics':vietocr_results},
    ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('VietOCR model and metrics saved:',weights_path,metrics_path,flush=True)
